# THB Rate: BOT vs. Yahoo Finance

Compares Bank of Thailand's officially published USD/THB rate against Yahoo Finance's `THB=X` spot quote over the last 1 year.

**Why not BOT's literal "Spot Rate" endpoint?** `src/bot_api`'s `spot_rate.daily()` (`Stat-SpotRate/v2/SPOTRATE`) is the BOT endpoint literally named "spot rate," but it's confirmed discontinued: it authenticates and returns the right shape, but every value field is blank (BOT's own response metadata marks the table discontinued, `last_updated: 2024-12-27` — see `notebooks/BOT_query.ipynb` §9).

**Why not the catalog's `usd_thb_bot` series?** That series also targets a BOT rate (`mid_rate`), but goes through `macro_data.sources.bot`, which calls the *old* `apigw1.bot.or.th` gateway — and that gateway is now fully dead (its hostname doesn't even resolve in DNS).

This notebook instead uses the standalone `bot_api` client's `reference_rate.daily()` — BOT's **weighted-average interbank reference rate** (THB/USD), on BOT's *new* gateway (`gateway.api.bot.or.th`), confirmed live in `notebooks/BOT_query.ipynb` §5.

Sections:
1. Aligning the two series
2. Comparison table
3. BOT vs. Yahoo: THB rate over the last year
4. Difference and correlation

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "src"))

import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

from bot_api import BOTClient
from macro_data import update, load

load_dotenv(Path("..") / ".env")
pd.set_option("display.max_rows", 20)
%matplotlib inline

bot_client = BOTClient()


def fetch_bot_reference_rate(client, start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    """reference_rate.daily() caps each call at a 31-day window; page through and concatenate."""
    frames = []
    chunk_start = start
    while chunk_start <= end:
        chunk_end = min(chunk_start + pd.Timedelta(days=30), end)
        frames.append(
            client.reference_rate.daily(chunk_start.strftime("%Y-%m-%d"), chunk_end.strftime("%Y-%m-%d"))
        )
        chunk_start = chunk_end + pd.Timedelta(days=1)
    return pd.concat(frames, ignore_index=True)


end_date = pd.Timestamp.today().normalize()
start_date = end_date - pd.Timedelta(days=365)

bot = fetch_bot_reference_rate(bot_client, start_date, end_date)
bot = bot.set_index("period").rename_axis("date").sort_index()
bot["rate"] = bot["rate"].astype(float)

yahoo_status = update("yahoo")  # refreshes all yahoo series; we only use usd_thb below
print("yahoo:", yahoo_status)

yahoo_full = load("usd_thb")
cutoff_yahoo = yahoo_full.index.max() - pd.Timedelta(days=365)
yahoo = yahoo_full[yahoo_full.index >= cutoff_yahoo]

print(f"BOT reference_rate: {len(bot)} rows, {bot.index.min().date()} to {bot.index.max().date()}")
print(f"Yahoo usd_thb: {len(yahoo)} rows, {yahoo.index.min().date()} to {yahoo.index.max().date()}")

$^GSPC: possibly delisted; no price data found  (1d 2026-07-15 -> 2026-07-14) (Yahoo error = "Invalid input - start date cannot be after end date. startDate = 1784088000, endDate = 1784081870")



1 Failed download:


['^GSPC']: possibly delisted; no price data found  (1d 2026-07-15 -> 2026-07-14) (Yahoo error = "Invalid input - start date cannot be after end date. startDate = 1784088000, endDate = 1784081870")


$^IRX: possibly delisted; no price data found  (1d 2026-07-15 -> 2026-07-14) (Yahoo error = "Invalid input - start date cannot be after end date. startDate = 1784091600, endDate = 1784081871")



1 Failed download:


...

['^TNX']: possibly delisted; no price data found  (1d 2026-07-15 -> 2026-07-14) (Yahoo error = "Invalid input - start date cannot be after end date. startDate = 1784091600, endDate = 1784081872")


$^TYX: possibly delisted; no price data found  (1d 2026-07-15 -> 2026-07-14) (Yahoo error = "Invalid input - start date cannot be after end date. startDate = 1784091600, endDate = 1784081873")



1 Failed download:


['^TYX']: possibly delisted; no price data found  (1d 2026-07-15 -> 2026-07-14) (Yahoo error = "Invalid input - start date cannot be after end date. startDate = 1784091600, endDate = 1784081873")


yahoo: {'usd_thb': 'up-to-date', 'eur_usd': 'up-to-date', 'gold_usd': 'up-to-date', 'brent_oil': 'up-to-date', 'sp500': 'up-to-date', 'set_index': 'up-to-date', 'usd_dxy': 'up-to-date', 'eur_thb': 'up-to-date', 'jpy_thb': 'up-to-date', 'gbp_thb': 'up-to-date', 'us_3m_yield': 'up-to-date', 'us_5y_yield': 'up-to-date', 'us_10y_yield': 'up-to-date', 'us_30y_yield': 'up-to-date'}
BOT reference_rate: 242 rows, 2025-07-15 to 2026-07-14
Yahoo usd_thb: 260 rows, 2025-07-15 to 2026-07-15


## 1. Aligning the two series

BOT publishes its reference rate on business days only; Yahoo's `THB=X` trades most calendar days (it's an OTC spot quote, not exchange-cleared). Inner-joining on date keeps only the days both sources published a value.

In [2]:
merged = pd.merge(
    bot.rename(columns={"rate": "bot_ref_rate"}),
    yahoo.rename(columns={"value": "yahoo_close"}),
    left_index=True,
    right_index=True,
    how="inner",
)

print(f"BOT dates: {len(bot)}, Yahoo dates: {len(yahoo)}, matched (inner join): {len(merged)}")
merged.tail()

BOT dates: 242, Yahoo dates: 260, matched (inner join): 241


,bot_ref_rate,yahoo_close
date,,
2026-07-08,33.393,33.360001
2026-07-09,33.465,33.470001
2026-07-10,33.318,33.299999
2026-07-13,33.376,33.330002
2026-07-14,33.508,33.509998


## 2. Comparison table

`abs_diff` is Yahoo's close minus BOT's reference rate (positive means Yahoo quotes more THB per USD than BOT's official average that day); `pct_diff` is the same difference as a percentage of BOT's rate.

In [3]:
comparison = merged.copy()
comparison["abs_diff"] = comparison["yahoo_close"] - comparison["bot_ref_rate"]
comparison["pct_diff"] = comparison["abs_diff"] / comparison["bot_ref_rate"] * 100

comparison.tail(10)

,bot_ref_rate,yahoo_close,abs_diff,pct_diff
date,,,,
2026-07-01,33.341,33.259998,-0.081002,-0.242949
2026-07-02,33.301,33.320000,0.019000,0.057054
2026-07-03,33.165,33.189999,0.024999,0.075377
2026-07-06,33.266,33.240002,-0.025998,-0.078153
2026-07-07,33.240,33.266998,0.026998,0.081222
2026-07-08,33.393,33.360001,-0.032999,-0.098821
2026-07-09,33.465,33.470001,0.005001,0.014945
2026-07-10,33.318,33.299999,-0.018001,-0.054027
2026-07-13,33.376,33.330002,-0.045998,-0.137818
